# How to Use the Tiles API

## Setup

In [ ]:
from datetime import datetime, timezone

import earthaccess
import httpx
from folium import Map, TileLayer

titiler_endpoint = "https://openveda.cloud/api/titiler-cmr"  # production endpoint

## Identify the dataset

You can find the MUR SST dataset using the `earthaccess.search_datasets` function.

In [ ]:
datasets = earthaccess.search_datasets(doi="10.5067/GHGMR-4FJ04")
ds = datasets[0]

concept_id = ds["meta"]["concept-id"]
print("Concept-Id: ", concept_id)

print("Abstract: ", ds["umm"]["Abstract"])

## Explore the collection using the `/compatibility` endpoint

See [How to use the Compatibility API Endpoint](./compatibility_api_example.ipynb)

## Define a query for titiler-cmr

To use titiler-cmr's endpoints for a NetCDF dataset like this we need to define a date range for the CMR query and a `variable` to analyze.

In [ ]:
variable = "sea_ice_fraction"
datetime_ = datetime(2024, 10, 10, tzinfo=timezone.utc).isoformat()

## Display tiles in an interactive map

The `/tilejson.json` endpoint will provide a parameterized `xyz` tile URL that can be added to an interactive map.

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/WebMercatorQuad/tilejson.json",
    params=(
        ("concept_id", concept_id),
        # Datetime in form of `start_date/end_date`
        ("datetime", datetime_),
        # titiler-cmr can work with both Zarr and COG dataset
        # but we need to tell the endpoints in advance which backend
        # to use
        ("backend", "xarray"),
        ("variable", variable),
        # We need to set min/max zoom because we don't want to use lowerzoom level (e.g 0)
        # which will results in useless large scale query
        ("minzoom", 2),
        ("maxzoom", 13),
        ("rescale", "0,1"),
        ("colormap_name", "blues_r"),
    ),
    timeout=None,
).json()

print(r)

In [ ]:
bounds = r["bounds"]
m = Map(location=(70, -40), zoom_start=3)

TileLayer(
    tiles=r["tiles"][0],
    opacity=1,
    attr="NASA",
).add_to(m)
m